<a href="https://colab.research.google.com/github/Sithranjan-Suresh/Team_ASCEND/blob/ascend-v2/ASCEND_Stage1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ASCEND Stage 1 — Held-out re-evaluation of V1

Reruns the existing GRPO pipeline (`ladder_optimizer_v4` + `variant_generator_v2`, unchanged) but evaluates pre/post accuracy on a **held-out** split instead of the training split. This is the first result in the Version 2 continuation — see the project doc for the full audit and roadmap.

Runs on a free Colab T4 GPU. Checkpoints go to Google Drive so a session disconnect doesn't lose progress; the training script itself resumes automatically (`RESUME_INTERACTIVE=0`).

## 1. Mount Drive (for checkpoint persistence across sessions)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/ascend_v2_checkpoints'
import os
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

Mounted at /content/drive


## 2. Clone your fork (ascend-v2 branch)

In [2]:
FORK_URL = 'https://github.com/Sithranjan-Suresh/Team_ASCEND.git'

!git clone --branch ascend-v2 $FORK_URL /content/ascend
%cd /content/ascend

# Restore checkpoints from a previous Colab session, if any exist.
!mkdir -p training_runs_holdout
!cp -r "$DRIVE_CHECKPOINT_DIR"/* training_runs_holdout/ 2>/dev/null || echo 'No prior checkpoints found on Drive -- starting fresh.'

Cloning into '/content/ascend'...
remote: Enumerating objects: 51879, done.
remote: Counting objects: 100% (51879/51879), done.
remote: Compressing objects: 100% (26088/26088), done.
remote: Total 51879 (delta 24672), reused 51839 (delta 24647), pack-reused 0 (from 0)
Receiving objects: 100% (51879/51879), 9.62 MiB | 16.31 MiB/s, done.
Resolving deltas: 100% (24672/24672), done.
Updating files: 100% (71640/71640), done.
/content/ascend


## 3. Install dependencies

In [3]:
# IMPORTANT: do NOT upgrade torch here -- Colab's preinstalled torch is already matched
# to its preinstalled torchvision/torchaudio and CUDA build. Upgrading torch alone (pip -U)
# breaks that binary match (surfaces later as 'RuntimeError: operator torchvision::nms
# does not exist' when transformers/peft import). Only install what Colab lacks.
#
# torchao IS upgraded on its own: current peft checks its version even when it's unused
# (LoRA dispatch probes it unconditionally) and errors on Colab's old preinstalled 0.10.0
# ('ImportError: Found an incompatible version of torchao ... only versions above 0.16.0
# are supported'). torchao is independent of torch/torchvision's compiled ops, so
# upgrading just it is safe.
!pip install -q transformers datasets accelerate peft trl bitsandbytes
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 45.5 MB/s eta 0:00:00


## 4. Build the stratified train/held-out split

Stratified by P3's `module` field (18 puzzle categories). Fixed seed → reproducible split, and it's saved into the repo so every later experiment (E1–E8 in the doc) can reuse the exact same held-out set.

In [4]:
!python split_dataset.py --seed 42 --heldout-frac 0.2

Total puzzles: 1715
Train: 1371  Held-out: 344
Wrote split to: PythonProgrammingPuzzles/puzzles/split_seed42.json


## 5. Run training + held-out evaluation

`TRAIN_LIMIT` / `HELDOUT_EVAL_LIMIT` default to 100 each, matching V1's `main_v2.py` scale. Lower these first if you just want a quick smoke test (e.g. 10/10) before committing a full T4 session to it.

In [ ]:
import os
os.environ['SPLIT_SEED'] = '42'
os.environ['TRAIN_LIMIT'] = '10'
os.environ['HELDOUT_EVAL_LIMIT'] = '10'
os.environ['RESUME_INTERACTIVE'] = '0'

!python run_holdout_eval.py

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Total puzzles: 1715  (train split: 1371, held-out split: 344)
Found incomplete run 00001 with 0/10 completed problems.
RESUME_INTERACTIVE=0 -> auto-resuming (non-interactive mode).
Run ID: 00001 (resumed)
config.json: 100% 631/631 [00:00<00:00, 3.15MB/s]
tokenizer_config.json: 100% 1.87k/1.87k [00:00<00:00, 5.37MB/s]
tokenizer.json: 100% 1.37M/1.37M [00:00<00:00, 45.8MB/s]

model.safetensors: downloading bytes:   0% 0.00/2.69G [00:00<?, ?B/s]
model.safetensors: downloading bytes:  11% 293M/2.69G [00:02<00:11, 208MB/s, 25.1MB/s  ]
model.safetensors: downloading bytes:  14% 365M/2.69G 

## 6. Back up checkpoints to Drive

Run this periodically during a long training loop (Colab can disconnect at any time), and again at the end.

In [ ]:
!cp -r training_runs_holdout/* "$DRIVE_CHECKPOINT_DIR"/
print('Checkpoints backed up to', DRIVE_CHECKPOINT_DIR)

## 7. Read back the result

Compare `pretrain_holdout` / `posttrain_holdout` here against the training-set numbers in the original `result*.txt` files on `integration-v1`. A much smaller (or vanished) gap on held-out data means V1's reported pass@k improvement was largely training-set memorization.

In [ ]:
import json, glob
run_dirs = sorted(glob.glob('training_runs_holdout/*/run.json'))
for rd in run_dirs:
    with open(rd) as f:
        data = json.load(f)
    print(rd)
    print('  status:', data.get('status'))
    print('  pretrain_holdout: ', data.get('results', {}).get('pretrain_holdout'))
    print('  posttrain_holdout:', data.get('results', {}).get('posttrain_holdout'))

## 8. (Optional) Push the split file + results back to your fork

Uses a GitHub personal access token, entered here and never written to disk or the repo — generate one at https://github.com/settings/tokens (repo scope) if you don't have one.

In [ ]:
import getpass
token = getpass.getpass('GitHub personal access token: ')

!git config user.email "sithranjan.suresh@gmail.com"
!git config user.name "Sithranjan Suresh"
!git add PythonProgrammingPuzzles/puzzles/split_seed42.json
!git commit -m "Stage 1: add stratified train/held-out split (seed 42)" || echo 'nothing to commit'
!git push https://$token@github.com/Sithranjan-Suresh/Team_ASCEND.git ascend-v2

del token